In [ ]:

# Step 4.2 - Load Dataset for Machine Learning

import pandas as pd

# Load cleaned master dataset
df = pd.read_csv(
    "/content/master_sales_cleaned.csv",
    parse_dates=["engage_date", "close_date"]
)

# Verify dataset
print("Dataset Shape:", df.shape)

print("\nDeal Stage Distribution:")
print(df["deal_stage"].value_counts())

print("\nFirst 5 Records:")
display(df.head())


In [ ]:

# Step 4.3 - Prepare ML Training Dataset

# Select only completed opportunities
ml_data = df[
    df["deal_stage"].isin(["Won", "Lost"])
].copy()

# Create target variable
# Won = 1, Lost = 0
ml_data["won_flag"] = (
    ml_data["deal_stage"] == "Won"
).astype(int)

# Verify training dataset
print("MACHINE LEARNING DATASET")
print("=" * 45)

print("Total Training Records:", len(ml_data))

print("\nTarget Distribution:")
print(ml_data["won_flag"].value_counts())

print("\nTarget Meaning:")
print("1 = Won Deal")
print("0 = Lost Deal")

print("\nFirst 5 Records:")
display(
    ml_data[
        ["opportunity_id", "deal_stage", "won_flag"]
    ].head()
)


In [ ]:

# Step 4.4 - Select Machine Learning Features

# Select features available before deal closure
feature_columns = [
    "sales_agent",
    "product",
    "sector",
    "revenue",
    "employees",
    "office_location",
    "series",
    "sales_price",
    "manager",
    "regional_office"
]

# Create input features
X = ml_data[feature_columns].copy()

# Create target variable
y = ml_data["won_flag"].copy()

# Verify selected features
print("MACHINE LEARNING FEATURE SELECTION")
print("=" * 45)

print("Total Input Features:", X.shape[1])
print("Total Training Records:", X.shape[0])

print("\nSelected Features:")
print(X.columns.tolist())

print("\nTarget Distribution:")
print(y.value_counts())

print("\nMissing Values in Features:")
print(X.isnull().sum())


In [ ]:

# Step 4.5 - Prepare ML Preprocessing Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Categorical features
categorical_features = [
    "sales_agent",
    "product",
    "sector",
    "office_location",
    "series",
    "manager",
    "regional_office"
]

# Numerical features
numerical_features = [
    "revenue",
    "employees",
    "sales_price"
]

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

# Numerical preprocessing
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(
            strategy="median"
        ))
    ]
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features
        )
    ]
)

print("ML PREPROCESSING PIPELINE CREATED")
print("=" * 45)
print("Categorical Features:", len(categorical_features))
print("Numerical Features:", len(numerical_features))
print("Total Features:", len(categorical_features) + len(numerical_features))
print("Preprocessor Ready:", preprocessor is not None)


In [ ]:

# Step 4.6 - Train-Test Split

from sklearn.model_selection import train_test_split

# Split features and target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Verify split
print("TRAIN-TEST SPLIT COMPLETED")
print("=" * 45)

print("Total Records:", len(X))

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())


In [ ]:

# Step 4.7 - Train Logistic Regression Model

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create the machine learning pipeline
win_prediction_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

# Train the model
win_prediction_model.fit(X_train, y_train)

print("SALESPULSE AI - MODEL TRAINING")
print("=" * 45)

print("Model: Logistic Regression")
print("Training Records:", len(X_train))
print("Input Features:", X_train.shape[1])
print("Model Training Completed Successfully!")


In [ ]:

# Step 4.8 - Evaluate Logistic Regression Model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

# Predict test data
y_pred = win_prediction_model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("SALESPULSE AI - MODEL EVALUATION")
print("=" * 45)

print(f"Accuracy: {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1 Score: {f1:.2%}")

print("\nCLASSIFICATION REPORT")
print("=" * 45)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Lost", "Won"],
    zero_division=0
))

# Display confusion matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Lost", "Won"]
)

disp.plot(cmap="Blues")
plt.title("SalesPulse AI - Confusion Matrix")
plt.show()


In [ ]:

# Step 4.9 - Compare Model with Baseline

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Create baseline model
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Train baseline
baseline_model.fit(X_train, y_train)

# Make predictions
baseline_pred = baseline_model.predict(X_test)

# Compare performance
print("MODEL PERFORMANCE COMPARISON")
print("=" * 50)

print("\nBASELINE MODEL")
print("Accuracy:",
      round(accuracy_score(y_test, baseline_pred) * 100, 2))
print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_test, baseline_pred) * 100, 2))

print("\nLOGISTIC REGRESSION")
print("Accuracy:",
      round(accuracy_score(y_test, y_pred) * 100, 2))
print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_test, y_pred) * 100, 2))

print("\nLost Deal Detection (F1 Score)")
print("Baseline:",
      round(f1_score(y_test, baseline_pred, pos_label=0), 3))
print("Logistic Regression:",
      round(f1_score(y_test, y_pred, pos_label=0), 3))


In [ ]:

# Step 4.10 - Balanced Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Create balanced model
balanced_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ]
)

# Train model
balanced_model.fit(X_train, y_train)

# Predict test data
balanced_pred = balanced_model.predict(X_test)

# Evaluate model
print("BALANCED LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print(
    "Accuracy:",
    round(accuracy_score(y_test, balanced_pred) * 100, 2)
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(y_test, balanced_pred) * 100,
        2
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        balanced_pred,
        target_names=["Lost", "Won"],
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, balanced_pred))


In [ ]:

# Step 4.11 - Random Forest Model

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Create Random Forest pipeline
random_forest_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train Random Forest
random_forest_model.fit(X_train, y_train)

# Predict test data
rf_pred = random_forest_model.predict(X_test)

# Evaluate performance
print("RANDOM FOREST MODEL RESULTS")
print("=" * 50)

print(
    "Accuracy:",
    round(accuracy_score(y_test, rf_pred) * 100, 2)
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(y_test, rf_pred) * 100,
        2
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=["Lost", "Won"],
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, rf_pred))


In [ ]:

# Step 4.12 - Random Forest Feature Importance

import pandas as pd
import matplotlib.pyplot as plt

# Get trained Random Forest
rf_classifier = random_forest_model.named_steps[
    "classifier"
]

# Get fitted preprocessor
rf_preprocessor = random_forest_model.named_steps[
    "preprocessing"
]

# Get transformed feature names
feature_names = rf_preprocessor.get_feature_names_out()

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_classifier.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

# Display top 15 features
print("TOP 15 IMPORTANT FEATURES")
print("=" * 50)

print(feature_importance.head(15).to_string(index=False))

# Plot top 15 features
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"],
    color="steelblue"
)

plt.gca().invert_yaxis()

plt.title("SalesPulse AI - Random Forest Feature Importance")
plt.xlabel("Feature Importance")
plt.ylabel("Features")

plt.tight_layout()
plt.show()


In [ ]:

# Step 4.13 - Analyze Deal Progress Features

print("SALESPULSE AI - FEATURE AVAILABILITY ANALYSIS")
print("=" * 55)

# Check date availability by deal stage
date_availability = (
    df.groupby("deal_stage")
    .agg(
        total_deals=("opportunity_id", "count"),
        engage_dates_available=(
            "engage_date",
            lambda x: x.notna().sum()
        ),
        close_dates_available=(
            "close_date",
            lambda x: x.notna().sum()
        )
    )
)

print("\nDATE AVAILABILITY BY DEAL STAGE")
print(date_availability)

# Check available deal stages
print("\nAVAILABLE DEAL STAGES")
print(df["deal_stage"].value_counts())

# Check date ranges
print("\nDATE RANGES")
print("Earliest Engagement:", df["engage_date"].min())
print("Latest Engagement:", df["engage_date"].max())
print("Earliest Closing:", df["close_date"].min())
print("Latest Closing:", df["close_date"].max())

# Check whether deal activity columns exist
activity_columns = [
    "last_activity_date",
    "followup_date",
    "interaction_count",
    "deal_amount",
    "stage_change_date"
]

print("\nSALES ACTIVITY FEATURES")
for column in activity_columns:
    print(
        f"{column}:",
        "Available" if column in df.columns
        else "Not Available"
    )


In [ ]:

# Step 4.14 - Historical Sales Cycle Analysis

# Use completed historical deals
cycle_data = ml_data.copy()

# Calculate days between engagement and closing
cycle_data["sales_cycle_days"] = (
    cycle_data["close_date"] -
    cycle_data["engage_date"]
).dt.days

# Remove invalid durations
cycle_data = cycle_data[
    cycle_data["sales_cycle_days"].notna()
    & (cycle_data["sales_cycle_days"] >= 0)
]

print("SALESPULSE AI - SALES CYCLE ANALYSIS")
print("=" * 50)

print("\nSALES CYCLE BY DEAL OUTCOME")

print(
    cycle_data.groupby("deal_stage")[
        "sales_cycle_days"
    ].agg(["count", "mean", "median", "min", "max"]).round(2)
)

print("\nOVERALL SALES CYCLE")

print(
    cycle_data["sales_cycle_days"]
    .describe()
    .round(2)
)

print("\nDEALS EXCEEDING 60 DAYS")

print(
    pd.crosstab(
        cycle_data["deal_stage"],
        cycle_data["sales_cycle_days"] > 60,
        normalize="index"
    ).mul(100).round(2)
)


In [ ]:

# Step 4.15 - Sales Cycle Win Rate Analysis

import pandas as pd

# Create historical sales cycle groups
cycle_data["cycle_group"] = pd.cut(
    cycle_data["sales_cycle_days"],
    bins=[0, 15, 30, 60, 90, float("inf")],
    labels=[
        "1-15 Days",
        "16-30 Days",
        "31-60 Days",
        "61-90 Days",
        "90+ Days"
    ]
)

# Calculate won and lost deals
cycle_summary = pd.crosstab(
    cycle_data["cycle_group"],
    cycle_data["deal_stage"]
)

# Calculate total deals
cycle_summary["Total Deals"] = (
    cycle_summary["Won"] + cycle_summary["Lost"]
)

# Calculate historical win rate
cycle_summary["Win Rate (%)"] = (
    cycle_summary["Won"]
    / cycle_summary["Total Deals"] * 100
).round(2)

print("SALESPULSE AI - SALES CYCLE WIN RATE")
print("=" * 55)

print(cycle_summary.to_string())


In [ ]:

# Step 4.16 - Open Sales Opportunities Analysis

import pandas as pd

# Select currently open opportunities
open_deals = df[
    df["deal_stage"].isin(["Engaging", "Prospecting"])
].copy()

print("SALESPULSE AI - OPEN OPPORTUNITIES")
print("=" * 55)

print("\nTOTAL OPEN OPPORTUNITIES")
print(len(open_deals))

print("\nOPEN OPPORTUNITIES BY STAGE")
print(open_deals["deal_stage"].value_counts())

print("\nMISSING VALUES IN KEY FEATURES")

key_features = [
    "engage_date",
    "account",
    "revenue",
    "employees",
    "sales_price",
    "sales_agent",
    "product",
    "regional_office"
]

print(
    open_deals[key_features]
    .isna()
    .sum()
)

print("\nOPEN OPPORTUNITIES BY REGION")
print(
    open_deals["regional_office"]
    .value_counts()
)

print("\nOPEN OPPORTUNITIES BY PRODUCT")
print(
    open_deals["product"]
    .value_counts()
)


In [ ]:

# Step 4.17 - Historical Product Win Rates

import pandas as pd

# Select completed deals
completed_deals = df[
    df["deal_stage"].isin(["Won", "Lost"])
].copy()

# Calculate won and lost counts by product
product_summary = pd.crosstab(
    completed_deals["product"],
    completed_deals["deal_stage"]
)

# Calculate total completed deals
product_summary["Total Deals"] = (
    product_summary["Won"] +
    product_summary["Lost"]
)

# Calculate historical win rate
product_summary["Win Rate (%)"] = (
    product_summary["Won"] /
    product_summary["Total Deals"] * 100
).round(2)

# Sort products by historical win rate
product_summary = product_summary.sort_values(
    "Win Rate (%)",
    ascending=False
)

print("SALESPULSE AI - PRODUCT WIN RATE ANALYSIS")
print("=" * 55)

print(product_summary.to_string())

# Overall historical win rate
overall_win_rate = (
    completed_deals["deal_stage"].eq("Won").mean() * 100
)

print("\nOVERALL HISTORICAL WIN RATE")
print(round(overall_win_rate, 2), "%")


In [ ]:

# Step 4.18 - Historical Regional Win Rates

import pandas as pd

# Calculate won and lost deals by region
region_summary = pd.crosstab(
    completed_deals["regional_office"],
    completed_deals["deal_stage"]
)

# Calculate total completed deals
region_summary["Total Deals"] = (
    region_summary["Won"] +
    region_summary["Lost"]
)

# Calculate historical win rate
region_summary["Win Rate (%)"] = (
    region_summary["Won"] /
    region_summary["Total Deals"] * 100
).round(2)

# Sort regions by win rate
region_summary = region_summary.sort_values(
    "Win Rate (%)",
    ascending=False
)

print("SALESPULSE AI - REGIONAL WIN RATE ANALYSIS")
print("=" * 55)

print(region_summary.to_string())

print("\nOVERALL HISTORICAL WIN RATE")
print(round(overall_win_rate, 2), "%")


In [ ]:

# Step 4.19 - Historical Industry Win Rates

import pandas as pd

# Calculate won and lost deals by industry
industry_summary = pd.crosstab(
    completed_deals["sector"],
    completed_deals["deal_stage"]
)

# Calculate total completed deals
industry_summary["Total Deals"] = (
    industry_summary["Won"] +
    industry_summary["Lost"]
)

# Calculate historical win rate
industry_summary["Win Rate (%)"] = (
    industry_summary["Won"] /
    industry_summary["Total Deals"] * 100
).round(2)

# Sort industries by win rate
industry_summary = industry_summary.sort_values(
    "Win Rate (%)",
    ascending=False
)

print("SALESPULSE AI - INDUSTRY WIN RATE ANALYSIS")
print("=" * 55)

print(industry_summary.to_string())

print("\nOVERALL HISTORICAL WIN RATE")
print(round(overall_win_rate, 2), "%")


In [ ]:

# Step 4.20 - Prepare Open Deals for Risk Assessment

import pandas as pd

# Select open opportunities
risk_data = df[
    df["deal_stage"].isin(["Engaging", "Prospecting"])
].copy()

# Flag unknown customer accounts
risk_data["unknown_account_flag"] = (
    risk_data["account"].isna()
    | risk_data["account"].eq("Unknown Account")
).astype(int)

# Flag missing customer information
risk_data["missing_customer_info_flag"] = (
    risk_data["revenue"].isna()
    | risk_data["employees"].isna()
).astype(int)

# Flag missing engagement dates
risk_data["missing_engagement_date_flag"] = (
    risk_data["engage_date"].isna()
).astype(int)

print("SALESPULSE AI - RISK DATA PREPARATION")
print("=" * 55)

print("\nTOTAL OPEN DEALS")
print(len(risk_data))

print("\nDATA QUALITY FLAGS")
print(
    risk_data[
        [
            "unknown_account_flag",
            "missing_customer_info_flag",
            "missing_engagement_date_flag"
        ]
    ].sum()
)

print("\nSAMPLE RECORDS")
print(
    risk_data[
        [
            "opportunity_id",
            "deal_stage",
            "product",
            "regional_office",
            "unknown_account_flag",
            "missing_customer_info_flag",
            "missing_engagement_date_flag"
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

# Step 4.21 - Explainable Data Quality Scoring

# Create a data quality issue score
risk_data["data_quality_score"] = (
    risk_data["unknown_account_flag"] * 30
    + risk_data["missing_customer_info_flag"] * 20
    + risk_data["missing_engagement_date_flag"] * 20
)

# Assign data quality review priority
def assign_review_priority(score):
    if score >= 50:
        return "High"
    elif score >= 20:
        return "Medium"
    else:
        return "Low"

risk_data["review_priority"] = (
    risk_data["data_quality_score"]
    .apply(assign_review_priority)
)

print("SALESPULSE AI - DATA QUALITY SCORING")
print("=" * 55)

print("\nREVIEW PRIORITY DISTRIBUTION")
print(
    risk_data["review_priority"]
    .value_counts()
)

print("\nDATA QUALITY SCORE SUMMARY")
print(
    risk_data["data_quality_score"]
    .describe()
)

print("\nSAMPLE SCORED OPPORTUNITIES")
print(
    risk_data[
        [
            "opportunity_id",
            "deal_stage",
            "data_quality_score",
            "review_priority"
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

# Step 4.22 - Explainable Recommendation Engine

def generate_recommendation(row):

    recommendations = []

    # Check customer account
    if row["unknown_account_flag"] == 1:
        recommendations.append(
            "Verify and update customer account details"
        )

    # Check customer information
    if row["missing_customer_info_flag"] == 1:
        recommendations.append(
            "Complete customer revenue and employee information"
        )

    # Check engagement date
    if row["missing_engagement_date_flag"] == 1:
        recommendations.append(
            "Record the customer engagement date"
        )

    # No data quality issues
    if not recommendations:
        recommendations.append(
            "Customer information is complete; review next sales action"
        )

    return " | ".join(recommendations)


# Generate recommendations
risk_data["ai_recommendation"] = risk_data.apply(
    generate_recommendation,
    axis=1
)

print("SALESPULSE AI - RECOMMENDATION ENGINE")
print("=" * 55)

print("\nTOTAL OPPORTUNITIES PROCESSED")
print(len(risk_data))

print("\nRECOMMENDATIONS BY PRIORITY")

for priority in ["High", "Medium", "Low"]:

    count = (
        risk_data["review_priority"] == priority
    ).sum()

    print(f"{priority}: {count}")

print("\nSAMPLE RECOMMENDATIONS")

print(
    risk_data[
        [
            "opportunity_id",
            "review_priority",
            "data_quality_score",
            "ai_recommendation"
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

# Step 4.23 - Automated Decision Logging

import pandas as pd
from datetime import datetime, timezone

# Record when the assessment was performed
assessment_timestamp = datetime.now(
    timezone.utc
).isoformat(timespec="seconds")

# Create decision log
decision_logs = risk_data[
    [
        "opportunity_id",
        "deal_stage",
        "sales_agent",
        "product",
        "regional_office",
        "data_quality_score",
        "review_priority",
        "ai_recommendation"
    ]
].copy()

# Add audit information
decision_logs["assessment_timestamp_utc"] = (
    assessment_timestamp
)

decision_logs["decision_type"] = (
    "Data Quality Assessment"
)

decision_logs["decision_method"] = (
    "Rules-Based"
)

decision_logs["rule_version"] = "1.0"

# Display results
print("SALESPULSE AI - AUTOMATED DECISION LOGS")
print("=" * 55)

print("\nTOTAL DECISION LOGS")
print(len(decision_logs))

print("\nDECISIONS BY PRIORITY")
print(
    decision_logs["review_priority"].value_counts()
)

print("\nSAMPLE DECISION LOGS")
print(
    decision_logs.head(5).to_string(index=False)
)

print("\nDUPLICATE OPPORTUNITY IDS")
print(
    decision_logs["opportunity_id"].duplicated().sum()
)


In [ ]:

# Step 4.24 - Export Sales Intelligence Results

import os

# Create output folder in Colab
output_folder = "/content/SalesPulse_AI_Outputs"

os.makedirs(output_folder, exist_ok=True)

# Define file paths
risk_file = os.path.join(
    output_folder,
    "sales_risk_assessment.csv"
)

logs_file = os.path.join(
    output_folder,
    "sales_decision_logs.csv"
)

# Export risk assessment results
risk_data.to_csv(
    risk_file,
    index=False
)

# Export decision logs
decision_logs.to_csv(
    logs_file,
    index=False
)

# Verify exported files
risk_export = pd.read_csv(risk_file)
logs_export = pd.read_csv(logs_file)

print("SALESPULSE AI - EXPORT VERIFICATION")
print("=" * 55)

print("\nRISK ASSESSMENT FILE")
print("File:", risk_file)
print("Rows:", len(risk_export))

print("\nDECISION LOG FILE")
print("File:", logs_file)
print("Rows:", len(logs_export))

print("\nEXPORT STATUS: SUCCESS")


In [ ]:

# Step 4.25 - Download SalesPulse AI Results

from google.colab import files

files.download(
    "/content/SalesPulse_AI_Outputs/sales_risk_assessment.csv"
)

files.download(
    "/content/SalesPulse_AI_Outputs/sales_decision_logs.csv"
)


In [ ]:

from google.colab import files

files.download("/content/SalesPulse_AI_Outputs/sales_decision_logs.csv")
